In [ ]:
import os
import sys
from loguru import logger
from datetime import datetime
import uuid
from elasticsearch import Elasticsearch, helpers
from typing import Optional, List, Dict, Any
from pymongo import MongoClient
from concurrent.futures import ThreadPoolExecutor, as_completed
path = r"C:\Users\Admin\Documents\V03-120126"
sys.path.append(path)
from constants import ElasticConfig, MongoDBConfig, MigrateConfig, MongoDBCollectionConfig

#Index name
v03_index = "document_data_301225_v0"
th_index = "document_data_281125"
target_index = "document_data_v1"
es = Elasticsearch([ElasticConfig.ELASTIC_HOST])
# Check if the Elasticsearch instance is reachable
if not es.ping():
    raise ConnectionError("Failed to connect to Elasticsearch at {}".format(ElasticConfig.ELASTIC_HOST))

In [ ]:
# Function to update createdBy field
def update_index_creator(es_client, index_name, creator_value):
    """
    Update createdBy field for all documents in the index where it is not 'UPLOAD' or 'upload'.
    """
    try:
        body = {
            "script": {
                "source": "ctx._source.createdBy = params.new_creator",
                "lang": "painless",
                "params": {
                    "new_creator": creator_value
                }
            },
            "query": {
                "bool": {
                    "must_not": [
                        {"terms": {"createdBy.keyword": ["UPLOAD", "upload"]}}
                    ]
                }
            }
        }

        logger.info(f"Starting update for index: {index_name}")
        response = es_client.update_by_query(
            index=index_name,
            body=body,
            wait_for_completion=True,
            request_timeout=300
        )
        logger.info(
            f"Updated createdBy for {index_name}. Updated: {response.get('updated')} docs."
        )

    except Exception as e:
        logger.error(f"Error updating {index_name}: {e}")


# Function to merge indices
def merge_indices(es_client, source_indices, target_index):
    """
    Merge multiple source indices into a target index.
    """
    try:
        logger.info(f"Merging indices {source_indices} into {target_index}...")

        body = {
            "source": {"index": source_indices},
            "dest": {"index": target_index}
        }

        response = es_client.reindex(
            body=body,
            wait_for_completion=True,
            request_timeout=3600
        )
        logger.info(f"Merge completed. stats: {response}")

    except Exception as e:
        logger.error(f"Error merging indices: {e}")


# ===== MULTI THREAD FUNCTIONS =====

def update_indices_multithread(es_client, indices, creator_value, max_workers=5):
    """
    Update createdBy for multiple indices using multi-threading
    """
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(update_index_creator, es_client, index, creator_value)
            for index in indices
        ]

        for future in as_completed(futures):
            future.result()  # raise exception if any


def merge_indices_multithread(es_client, merge_jobs, max_workers=3):
    """
    merge_jobs = [
        (["index1", "index2"], "target_index"),
        (["index3", "index4"], "target_index_2")
    ]
    """
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(merge_indices, es_client, sources, target)
            for sources, target in merge_jobs
        ]

        for future in as_completed(futures):
            future.result()


In [ ]:
# ====== UPDATE createdBy MULTI THREAD ======
indices_to_update = [
    # "document_data_301225_v0",
    "document_data_281125"
]

update_indices_multithread(
    es_client=es,
    indices=indices_to_update,
    creator_value="SYSTEM",  # "V03"
    max_workers=5
)

In [ ]:
# ====== MERGE INDICES MULTI THREAD ======
merge_jobs = [
    (["document_data_301225_v0", "document_data_281125"], "document_data_v1"),
]

merge_indices_multithread(
    es_client=es,
    merge_jobs=merge_jobs,
    max_workers=2
)
